### 224 REALHW 小红的置信约束训练损失
链接： [224 REALHW 小红的置信约束训练损失](https://www.nowcoder.com/practice/ee2c352b86424e0bb86b40436bb54268?tpId=390&tags=&title=&difficulty=0&judgeStatus=0&rp=0&sourceUrl=%2Fexam%2Foj%2Fta%3FtpId%3D37)

![描述](desc.png)

In [ ]:
# 我的初始解法
from os import curdir
import sys
from math import log, exp
from collections import deque, defaultdict

def softmax(logits: list):
    exp_x = list(map(exp, logits))
    sum_exp_x = sum(exp_x)
    return [ex / sum_exp_x for ex in exp_x]

data = sys.stdin.buffer.read().split()
it = iter(data)

N = int(next(it))
V = int(next(it))
B = int(next(it))
lam = float(next(it))

logits = []
probs = []
y = []
c = []
l = []
H = []
for i in range(N):
    logits.append([float(next(it)) for _ in range(V)])
    y.append(int(next(it)))
    c.append(int(next(it)))

    probs.append(softmax(logits[-1]))
    log_probs = list(map(log, probs[-1]))

    l.append(-log_probs[y[-1]])
    H.append(sum(list(map(lambda x: -x[0] * x[1], zip(probs[-1], log_probs)))))
# 01 背包求交叉熵最大值
# 状态表示：dp[i][j] 对于前i个物品，总代价不超过j的最大交叉熵
# 状态转移：
# 1. 选 i，dp[i - 1][j - c[i]] + l[i]
# 2. 不选 i，dp[i - 1][j]
dp = [[0] * (B + 2) for i in range(N + 2)]
g = defaultdict(list)
for i in range(N):
    for j in range(0, B + 1):
        a = dp[i - 1][j]
        if j >= c[i]:
            b = dp[i - 1][j - c[i]] + l[i]
            dp[i][j] = max(a, b)
        else:
            dp[i][j] = a

eps = 1e-9
cur_j = B
while cur_j and abs(dp[N - 1][cur_j - 1] - dp[N - 1][cur_j]) < eps:
    cur_j -= 1



res = []
def dfs(cur_i: int, cur_j: int, path: list):
    global res
    # if res is not None:
    #     return False
    if cur_i == -1 and cur_j == 0:
        res = path.copy()
        return True
    
    flag = False
    # 不选 i 转移过来
    if abs(dp[cur_i - 1][cur_j] - dp[cur_i][cur_j]) < eps:
        flag |= dfs(cur_i - 1, cur_j, path)
    # i - 1 选 i 转移过来
    elif abs(dp[cur_i - 1][cur_j - c[cur_i]] + l[cur_i] - dp[cur_i][cur_j]) < eps:
        path.append(cur_i)
        flag |= dfs(cur_i - 1, cur_j - c[cur_i], path)
        path.pop()
    return flag

dfs(N - 1, cur_j, [])

ans = sum(l)
for i in res:
    ans += lam * H[i]
sys.stdout.write(f"{ans:.2f}\n")


"""
通过全部用例
运行时间
1647ms
占用内存
95084KB
"""

踩坑：
1. 一开始没有更新 j < c[i] 时的 dp[i][j]，导致后续无法正确更新 dp[i][j]
2. 没有使用 eps 来避免浮点数精度问题，导致答案错误
3. 使用 DFS 查找路径耗时且想了很久，编码很慢
4. 没有理解题目中的基础损失之和是什么
    > 最终损失 = 所有样本的交叉熵损失之和 + 被选样本的分布熵惩罚
    >
    > 为什么需要所有样本的交叉熵损失之和？
    >
    > 因为每个样本都有自己的正确标签，不管这个样本有没有被选入集合，都需要对他的预测做优化————计算交叉熵损失
    > 
    > 为什么需要被选样本的分布熵惩罚？
    >
    > 题目中限制了代价预算 B，所以不能对所有样本都进行优化，所以要选一个集合 S
    > 
    > 选择标准是：在总代价不超过 B 的前提下，让被选样本的交叉熵之和最大。
    >
    > 也就是优先挑选那些模型预测更差的样本做优化，对他们施加额外的分布熵惩罚，鼓励模型在这些样本上有更准确的预测，而不是相对平均的概率预测。

In [ ]:
# 使用状态转移优化掉 DFS
from os import curdir
import sys
from math import log, exp
from collections import deque, defaultdict

eps = 1e-9
def fequal(a, b):
    return abs(a - b) < eps

def softmax(logits: list):
    exp_x = list(map(exp, logits))
    sum_exp_x = sum(exp_x)
    return [ex / sum_exp_x for ex in exp_x]

data = sys.stdin.buffer.read().split()
it = iter(data)

N = int(next(it))
V = int(next(it))
B = int(next(it))
lam = float(next(it))

logits = []
probs = []
y = []
c = []
l = []
H = []
for i in range(N):
    logits.append([float(next(it)) for _ in range(V)])
    y.append(int(next(it)))
    c.append(int(next(it)))

    probs.append(softmax(logits[-1]))
    log_probs = list(map(log, probs[-1]))

    l.append(-log_probs[y[-1]])
    H.append(sum(list(map(lambda x: -x[0] * x[1], zip(probs[-1], log_probs)))))
# 01 背包求交叉熵最大值
# 状态表示：dp[i][j] 对于前i个物品，总代价不超过j的最大交叉熵
# 状态转移：
# 1. 选 i，dp[i - 1][j - c[i]] + l[i]
# 2. 不选 i，dp[i - 1][j]
dp = [[0] * (B + 2) for i in range(N + 2)]
g = defaultdict(list)
for i in range(N):
    for j in range(0, B + 1):
        dp[i][j] = dp[i - 1][j]
        if j >= c[i]:
            dp[i][j] = max(dp[i][j], dp[i - 1][j - c[i]] + l[i])


cur_j = B
while cur_j and fequal(dp[N - 1][cur_j], dp[N - 1][cur_j - 1]):
    cur_j -= 1

selected = []
for i in range(N - 1, -1, -1):
    # 看看从 i - 1 是否需要选中 i 
    ci = c[i]
    li = l[i]
    if cur_j >= ci and fequal(dp[i - 1][cur_j - ci] + li, dp[i][cur_j]):
        selected.append(i)
        cur_j -= ci

ans = sum(l)
for i in selected:
    ans += lam * H[i]
sys.stdout.write(f"{ans:.2f}\n")


踩坑：
1. if cur_j >= ci and (dp[i - 1][cur_j - ci] + li - dp[i][cur_j]) < eps:
    > 回溯时判断物品是否被选中的条件写错了，缺少 abs